# MAI-Transcribe-1: Enterprise-Grade Speech-to-Text

> **Model Card:** [ai.azure.com/catalog/models/MAI-Transcribe-1](https://ai.azure.com/catalog/models/MAI-Transcribe-1)

MAI-Transcribe-1 is Microsoft AI's first-generation speech recognition model, designed for production-grade transcription across **25 languages**. It is powered by an autoregressive LLM-enhanced architecture that delivers:

| Attribute | Detail |
|---|---|
| **Architecture** | Autoregressive + text-prediction |
| **Languages** | 25 (Arabic, Chinese, Czech, Danish, Dutch, English, Finnish, French, German, Hindi, Hungarian, Indonesian, Italian, Japanese, Korean, Norwegian, Polish, Portuguese, Romanian, Russian, Spanish, Swedish, Thai, Turkish, Vietnamese) |
| **Input formats** | WAV · MP3 · FLAC (≤70 MB per file with mai-transcribe-1 model) |
| **API version** | `2025-10-15` |
| **Regions** | East US · West US |
| **Pricing** | **\$0.36 / audio hour** |
| **Powers** | Copilot Voice Mode, Dictation in Copilot, Azure Speech |

### What makes it stand out?
- **#1 on FLEURS benchmark** for 11 of the top-25 global languages
- Beats Whisper-large-v3 on the remaining 14 languages
- Beats Gemini 2.1 Flash on 11 of those 14 languages
- ~50% lower GPU cost than leading alternatives
- Robust to noisy real-world environments and diverse accents

## 1. Setup

Install dependencies and load environment variables from `.env`.

In [ ]:
# Install dependencies (run once)
%pip install -q requests python-dotenv azure-identity

In [ ]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

# Load credentials from .env file
load_dotenv()

TRANSCRIBE_SPEECH_KEY = os.getenv("TRANSCRIBE_SPEECH_KEY") or os.getenv("SPEECH_KEY")
TRANSCRIBE_SPEECH_REGION = os.getenv("TRANSCRIBE_SPEECH_REGION") or os.getenv("SPEECH_REGION", "eastus")  # eastus or westus

# Backward-compatible aliases used in later cells
SPEECH_KEY = TRANSCRIBE_SPEECH_KEY
SPEECH_REGION = TRANSCRIBE_SPEECH_REGION

assert TRANSCRIBE_SPEECH_KEY, "Set TRANSCRIBE_SPEECH_KEY (or SPEECH_KEY) in your .env file"
assert TRANSCRIBE_SPEECH_REGION, "Set TRANSCRIBE_SPEECH_REGION (or SPEECH_REGION) in your .env file"

# LLM Speech / Fast Transcription endpoint
TRANSCRIBE_URL = (
    f"https://{TRANSCRIBE_SPEECH_REGION}.api.cognitive.microsoft.com"
    "/speechtotext/transcriptions:transcribe"
    "?api-version=2025-10-15"
 )

HEADERS = {"Ocp-Apim-Subscription-Key": TRANSCRIBE_SPEECH_KEY}

print(f"✅ Endpoint : {TRANSCRIBE_URL}")
print(f"✅ Region   : {TRANSCRIBE_SPEECH_REGION}")

## 2. Helper: Use Local Audio Files

Use local WAV files from the folder configured in `.env` as `TRANSCRIBE_LOCAL_AUDIO_DIR`.
Example: `TRANSCRIBE_LOCAL_AUDIO_DIR=C:\Flutter\azure-transcription\demodata`
Set `AUDIO_FILE` manually if you want to force a specific file.

In [ ]:
from pathlib import Path
import os

LOCAL_AUDIO_DIR_ENV = os.getenv("TRANSCRIBE_LOCAL_AUDIO_DIR")
assert LOCAL_AUDIO_DIR_ENV, (
    "Set TRANSCRIBE_LOCAL_AUDIO_DIR in .env, e.g. C:\\Flutter\\azure-transcription\\demodata"
 )

LOCAL_AUDIO_DIR = Path(LOCAL_AUDIO_DIR_ENV).expanduser()
PREFERRED_FILES = [
    "sampledata_audiofiles_katiesteve.wav",
    "conversationrecording_new.wav",
]

assert LOCAL_AUDIO_DIR.exists(), f"Local audio folder not found: {LOCAL_AUDIO_DIR}"

available_wavs = sorted(LOCAL_AUDIO_DIR.glob("*.wav"))
assert available_wavs, f"No .wav files found in: {LOCAL_AUDIO_DIR}"

# Prefer known demo files, then fall back to the first WAV in the folder.
selected_audio = None
for name in PREFERRED_FILES:
    candidate = LOCAL_AUDIO_DIR / name
    if candidate.exists():
        selected_audio = candidate
        break
if selected_audio is None:
    selected_audio = available_wavs[0]

AUDIO_FILE = str(selected_audio)
file_size_mb = selected_audio.stat().st_size / (1024 * 1024)

print(f"✅ Using local audio: {AUDIO_FILE} ({file_size_mb:.2f} MB)")
print("Available WAV files:")
for wav in available_wavs:
    print(f" - {wav.name}")

## 3. Basic Transcription with MAI-Transcribe-1

The simplest call: send an audio file → receive a full transcript.

In [ ]:
def transcribe_audio(audio_path: str, extra_definition: dict | None = None) -> dict:
    """Transcribe an audio file using MAI-Transcribe-1."""
    definition = {
        "enhancedMode": {
            "enabled": True,
            "model": "mai-transcribe-1"
        }
    }
    if extra_definition:
        definition.update(extra_definition)

    with open(audio_path, "rb") as audio_file:
        response = requests.post(
            TRANSCRIBE_URL,
            headers=HEADERS,
            files={
                "audio":      (Path(audio_path).name, audio_file, "audio/wav"),
                "definition": (None, json.dumps(definition)),
            },
        )

    response.raise_for_status()
    return response.json()


result = transcribe_audio(AUDIO_FILE)

print("=" * 60)
print("TRANSCRIPT")
print("=" * 60)
for phrase in result.get("combinedPhrases", []):
    print(phrase["text"])

duration_ms = result.get("durationMilliseconds", 0)
print(f"\nAudio duration: {duration_ms / 1000:.2f}s")

## 4. Segment-Level Details (Words + Timestamps)

The `phrases` array gives per-segment detail with word-level timestamps.

In [ ]:
phrases = result.get("phrases", [])

for i, phrase in enumerate(phrases, 1):
    offset  = phrase.get("offsetMilliseconds", 0) / 1000
    dur     = phrase.get("durationMilliseconds", 0) / 1000
    text    = phrase.get("text", "")
    words   = phrase.get("words", [])

    print(f"[{offset:.2f}s – {offset + dur:.2f}s] {text}")
    for w in words[:5]:          # show first 5 words per phrase
        w_off = w["offsetMilliseconds"] / 1000
        w_dur = w["durationMilliseconds"] / 1000
        print(f"  {w['text']!r:20s}  @{w_off:.3f}s  ({w_dur*1000:.0f}ms)")
    if len(words) > 5:
        print(f"  ... and {len(words) - 5} more words")

## 5. Speaker Diarization

Identify who is speaking. Enable `diarization` in the request definition.

In [ ]:
# Note: diarization is NOT supported with mai-transcribe-1 model directly.
# Use the standard LLM speech (enhanced mode) for diarization.

diarization_definition = {
    "enhancedMode": {
        "enabled": True,
        "task": "transcribe"
        # model key omitted → uses default LLM speech (not mai-transcribe-1)
        # Add  "model": "mai-transcribe-1"  to use MAI model (diarization unsupported)
    },
    "diarization": {
        "enabled": True,
        "maxSpeakers": 2
    },
    "profanityFilterMode": "Masked"
}

diar_result = transcribe_audio(AUDIO_FILE, extra_definition=diarization_definition)

for phrase in diar_result.get("phrases", []):
    speaker = phrase.get("speaker", "?")
    offset  = phrase.get("offsetMilliseconds", 0) / 1000
    text    = phrase.get("text", "")
    print(f"Speaker {speaker}  [{offset:.2f}s]  {text}")

## 6. Prompt-Tuning for Custom Output

Guide output style with a prompt — enforce lexical format, highlight phrases, etc.

In [ ]:
# Lexical format: raw words without punctuation/capitalization
lexical_definition = {
    "enhancedMode": {
        "enabled": True,
        "model": "mai-transcribe-1",
        "task": "transcribe",
        "prompt": ["Output must be in lexical format."]
    }
}

lexical_result = transcribe_audio(AUDIO_FILE, extra_definition=lexical_definition)

print("Lexical output:")
for phrase in lexical_result.get("combinedPhrases", []):
    print(phrase["text"])

In [ ]:
# Highlight specific domain terms / acronyms for better recognition
domain_definition = {
    "enhancedMode": {
        "enabled": True,
        "model": "mai-transcribe-1",
        "task": "transcribe",
        "prompt": [
            "Pay attention to Azure, Microsoft Foundry, MAI, Copilot, LLM."
        ]
    }
}

domain_result = transcribe_audio(AUDIO_FILE, extra_definition=domain_definition)

print("Domain-tuned transcript:")
for phrase in domain_result.get("combinedPhrases", []):
    print(phrase["text"])

## 7. Translation to Another Language

LLM Speech can translate audio directly to a target language.

In [ ]:
# Translate audio to Korean
translate_definition = {
    "enhancedMode": {
        "enabled": True,
        "task": "translate",
        "targetLanguage": "ko"
        # Supported: en, zh, de, fr, it, ja, es, pt, ko
    }
}

translate_result = transcribe_audio(AUDIO_FILE, extra_definition=translate_definition)

print("Korean translation:")
for phrase in translate_result.get("combinedPhrases", []):
    print(phrase["text"])

## 8. Using Microsoft Entra ID Authentication (Production)

For production workloads, use managed identity instead of API keys.

In [ ]:
# Uncomment and run if USE_ENTRA_AUTH=true in your .env

# from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# token_provider = get_bearer_token_provider(
#     DefaultAzureCredential(),
#     "https://cognitiveservices.azure.com/.default"
# )
# entra_token = token_provider()

# entra_headers = {
#     "Authorization": f"Bearer {entra_token}",
# }
# # Then pass entra_headers instead of HEADERS to requests.post()

print("Entra ID auth snippet above — uncomment to use managed identity.")

## 9. 💰 Cost Calculator

**MAI-Transcribe-1 pricing: \$0.36 per audio hour**

This is approximately **50% lower** than comparable alternatives.

In [ ]:
# ── Cost Calculator ─────────────────────────────────────────
PRICE_PER_HOUR = 0.36   # USD per audio hour (as of April 2026)

# Edit the inputs below:
scenarios = {
    "1-hour call center recording": 1.0,
    "Full workday meetings (8 hrs)":   8.0,
    "Video archive batch (100 hrs)":  100.0,
    "Annual call center (~5000 hrs)": 5000.0,
}

print(f"{'Scenario':<45} {'Audio Hours':>12} {'Cost (USD)':>12}")
print("-" * 71)
for label, hours in scenarios.items():
    cost = hours * PRICE_PER_HOUR
    print(f"{label:<45} {hours:>12.1f} ${cost:>11.2f}")

print()
print("Comparison vs. leading alternatives (~$0.72/hr)")
alt_price = 0.72
print(f"{'Scenario':<45} {'Saving (USD)':>12} {'Saving %':>10}")
print("-" * 69)
for label, hours in scenarios.items():
    saving = hours * (alt_price - PRICE_PER_HOUR)
    pct    = (1 - PRICE_PER_HOUR / alt_price) * 100
    print(f"{label:<45} ${saving:>11.2f} {pct:>9.0f}%")

## 10. Summary & Next Steps

| Feature | Status |
|---|---|
| Basic transcription | ✅ |
| Word-level timestamps | ✅ |
| Speaker diarization | ✅ (LLM speech mode) |
| Prompt-tuning | ✅ |
| Translation | ✅ |
| Real-time transcription | 🔜 Coming soon |
| Context biasing | 🔜 Coming soon |

**Resources:**
- [Model Card](https://ai.azure.com/catalog/models/MAI-Transcribe-1)
- [LLM Speech API docs](https://learn.microsoft.com/azure/ai-services/speech-service/llm-speech)
- [Azure Speech regions](https://learn.microsoft.com/azure/ai-services/speech-service/regions)
- [MAI Playground](https://playground.microsoft.ai)